# 알약 이미지 EDA
개인 보강본입니다. 공용 notebooks/01_eda.ipynb는 변경하지 않습니다. 사진 비교는 설정 경로의 로컬 원본과 팀 교정표에서 다시 생성합니다.
2026-11-02 보강: 종류 교정 3건·검증 누락 박스 4건의 사진 근거와 재평가 연결을 추가했습니다.
충은님 EDA를 바탕으로 파일 정합성·누락 사진 대조 분석을 합친 통합본을 보완했습니다.
계산은 `src/eda.py`, 시각화는 `src/visualize.py`에 두고 노트북에서 호출합니다.
사진이 포함된 실행 출력은 팀 내부 검토용입니다. 공개 PR에는 출력 없는 노트북과 코드만 올립니다.

## 1. 데이터 배경과 출처
사진 속 알약의 위치와 종류를 찾는 프로젝트입니다.
출처: [Codeit AI 15기 Kaggle 대회 데이터](https://www.kaggle.com/competitions/ai15-level1-project/data).
원본(v0)을 분석하며, 정제본·합성 데이터의 수치와 구분합니다.

**발견한 점:** 학습 이미지와 테스트 이미지는 역할이 다릅니다.

**다음 행동:** 원본은 보존하고, 수정은 별도 데이터 버전으로 관리합니다.

## 2. EDA란?
모델에게 공부시킬 자료를 먼저 펼쳐 보는 작업입니다. 자료량, 정답 표시, 촬영 조건을 확인합니다.

**발견한 점:** 사진 수만으로는 자료 품질이나 모델 성능을 알 수 없습니다.

**다음 행동:** 통계에서 발견한 후보와 사진으로 확인한 사실을 구분합니다.

## 3. 데이터 읽기
설정 파일의 상대 경로를 사용합니다. 프로젝트 루트 또는 notebooks 폴더에서 실행합니다.
`data/raw/` 아래에 원본을 연결하고 모든 셀을 위에서부터 실행하세요.

**발견한 점:** 원본 사진 232장·JSON 763개·클래스 56개·박스 763개입니다.

**다음 행동:** 수치가 다르면 코드보다 데이터 경로와 버전을 먼저 확인합니다.


In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("프로젝트 폴더에서 실행하세요.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import eda as eda
from src import visualize as viz

viz.set_korean_font()
CONFIG = "configs/eda_soonwoo.yaml"
ctx = eda.load_eda_context(CONFIG)
display(eda.eda_inventory(ctx))


## 4. 무작위 사진 50장
10행 × 5열, seed 42로 같은 사진을 선택합니다.

**발견한 점:** 한 장에 여러 알약이 등장합니다. 무작위 50장은 전체 품질을 보증하는 검사가 아닙니다.

**다음 행동:** 아래 클래스별 사진과 오류 후보를 별도로 확인합니다.

In [ ]:
viz.plot_random_photos(ctx["image_dir"], n=50, rows=10)

## 5. 파일명 규칙
파일명 토큰과 JSON 메타데이터를 나란히 표시합니다.

**발견한 점:** 앞쪽 알약 코드와 각도 토큰을 대조할 수 있습니다. 배경·조명 토큰 전체의 공식 코드표는 미확보입니다.

**다음 행동:** 확인되지 않은 토큰에 색상·촬영 의미를 임의로 붙이지 않습니다.

In [ ]:
display(eda.filename_token_table(ctx))
display(eda.filename_metadata(ctx))

## 6. 클래스 분포
알약 종류별 정답 박스 수를 셉니다.

**발견한 점:** 원본에서 클래스별 박스 수는 최소 3개·최대 153개로 차이가 큽니다.

**다음 행동:** 드문 클래스를 더 자주 학습하는 방법을 같은 검증 분할에서 비교합니다.

In [ ]:
class_counts = eda.eda_class_counts(ctx)
display(class_counts)
viz.plot_eda_counts(class_counts,"label_idx","박스 수","라벨별 정답 박스 수")

## 7. 클래스별 알약 사진과 라벨 대조
정답 박스로 잘라 연속 라벨·원본 코드·이름을 함께 표시합니다.

**발견한 점:** category_id와 label_idx는 서로 다른 번호입니다. 대표 사진만으로 모든 라벨의 정확성을 확정할 수 없습니다.

**다음 행동:** 모양이 다른 사례는 원본 사진과 JSON까지 함께 검토합니다.

In [ ]:
viz.plot_class_crops(ctx)

## 8. 라벨 오류 의심 항목
파일명 차집합과 박스 경계·겹침 등을 검사합니다.

**발견한 점:** 원본에서 누락 후보 사진 8장이 나옵니다. 10/26 사진 대조에서는 각 사진에 박스 없는 알약 1개가 확인됐습니다.

**다음 행동:** 누락 코드의 약품 정체는 파일명 추정과 구분하고, 정제본 반영 여부는 별도로 확인합니다.

In [ ]:
missing = eda.filename_missing_candidates(ctx)
display(missing)
display(eda.find_label_issues(ctx["images"],ctx["boxes"],image_dir=ctx["image_dir"]))
viz.plot_missing_candidates(ctx, missing)

### 8.1 사진으로 확인한 종류 라벨 오류 3건
같은 사진의 각인과, 다른 사진의 같은 라벨 알약을 비교한 뒤 팀 `manual_fix.csv`와 대조했습니다.

**발견한 점:** 플라빅스→일양하이트린 1건과 아토젯↔트윈스타 2건은 이미 교정표에 등록돼 있습니다. 두 모델에서 공통으로 오분류처럼 집계된 사례입니다. 사진의 약품을 새로 감정한 결과가 아니라 데이터의 라벨·교정표 대조 결과입니다.

**다음 행동:** 데이터 정답 버전을 맞춘 뒤 모델 오류를 해석합니다. 아래 사진은 왼쪽부터 문제 영역, 원래 정답 이름의 참고 사진, 예측·교정 이름의 참고 사진입니다.


In [ ]:
quality_audit = eda.load_quality_audit(ctx, CONFIG)
display(eda.quality_audit_table(quality_audit, "labels"))
viz.plot_quality_audit(quality_audit, ctx, "labels")


### 8.2 교정표의 누락 박스 추가 4건
초록은 기존 정답 박스, 주황은 교정표에서 추가한 정답 박스입니다. 왼쪽은 교정 전, 오른쪽은 추가 후입니다.

**발견한 점:** 이번 검증 47장에 해당하는 교정표에는 종류 변경 3건과 박스 추가 4건이 있습니다. 박스 수는 153개에서 157개가 됩니다. 앞서 찾은 누락 후보 8장 전부를 해결했다는 뜻은 아닙니다.

**다음 행동:** 나머지 누락 후보의 처리·제외 여부를 별도로 확인합니다. 원본(v0)은 보존하고 교정본과 구분합니다.


In [ ]:
display(eda.quality_audit_table(quality_audit, "missing"))
viz.plot_quality_audit(quality_audit, ctx, "missing")


## 9. 사진과 JSON의 파일 대조
사진 목록과 JSON 이미지 목록을 양방향으로 비교합니다.

**발견한 점:** 원본에서 사진만 있거나 JSON만 있는 파일은 각각 0개입니다. 일부 알약의 박스 누락 8건과는 다른 검사입니다.

**다음 행동:** 새 버전에서도 파일 대조와 박스 대조를 각각 실행합니다.

In [ ]:
display(eda.file_alignment(ctx))

## 10. 촬영 배경별 이미지 수
배경·조명 메타데이터별 고유 사진 수입니다.

**발견한 점:** 메타데이터 코드별 수량은 확인되지만 실제 배경색을 보증하지 않습니다.

**다음 행동:** 성능과 배경의 관계를 볼 때 사진과 함께 확인합니다.

In [ ]:
background = eda.metadata_counts(ctx,"back_color")
display(background)
display(eda.metadata_counts(ctx,"light_color"))
viz.plot_eda_counts(background,"back_color","고유 사진 수","배경 코드별 고유 사진 수")

## 11. 알약 조합
파일명 기준 같은 알약 조합의 사진 수를 셉니다.

**발견한 점:** 232장에는 같은 조합의 다른 각도 사진이 포함되며, 파일명 기준 조합은 114개입니다.

**다음 행동:** 같은 조합이 학습·검증 양쪽에 겹치지 않도록 그룹 분할도 비교합니다.

In [ ]:
combinations = eda.pill_combinations(ctx)
display(combinations.head(20))
print("고유 조합 수:",len(combinations))

## 12. 촬영 각도
JSON 각도별 사진 수와 파일명 각도 일치를 봅니다.

**발견한 점:** 촬영 각도별 표본 수와 코드 일치 여부를 확인할 수 있습니다. 각도 토큰이 사진 품질을 의미하지는 않습니다.

**다음 행동:** 각도별 오류율은 모델 예측을 받은 뒤 같은 기준으로 계산합니다.

In [ ]:
angles = eda.metadata_counts(ctx,"camera_la")
display(angles)
display(eda.angle_alignment(ctx))
viz.plot_eda_counts(angles,"camera_la","고유 사진 수","촬영 각도별 고유 사진 수")

## 13. 이미지와 박스 상태
크기·알약 수·상대 박스 크기·파일 무결성을 확인합니다.

**발견한 점:** 원본 읽기 오류·메타데이터 크기 불일치·동일 바이트 중복 그룹은 0입니다. 작은 박스는 해상도 실험의 근거가 됩니다.

**다음 행동:** 경계 밖·겹침 후보는 사진으로 확인하고 자동 삭제하지 않습니다.

In [ ]:
display(eda.image_integrity(ctx))
display(eda.image_size_stats(ctx["images"]))
display(eda.objects_per_image(ctx["images"]))
display(eda.bbox_stats(ctx["boxes"]))
display(eda.class_box_size(ctx["boxes"]).head(10))
viz.plot_hist(ctx["boxes"]["rel_area"],"이미지 대비 박스 면적",xlabel="rel_area")
viz.plot_box_centers(ctx["boxes"])

## 14. 테스트 이미지 확인
정답 없는 테스트 사진을 학습과 별도로 검사합니다.

**발견한 점:** 테스트 842장의 읽기 오류는 0개입니다. 정답이 없어 이 분석만으로 정확도를 계산할 수 없습니다.

**다음 행동:** 테스트 사진을 학습 증강에 섞지 않고 최종 평가에 사용합니다.

In [ ]:
test_inventory = eda.test_image_inventory(ctx)
display(test_inventory)
viz.plot_random_photos(ctx["test_dir"], n=10)

## 15. 증강 전후 예시
좌우 반전 시 박스 좌표도 함께 변환합니다.

**발견한 점:** 이미지만 반전하면 기존 박스 좌표가 알약 위치와 어긋납니다. 아래 그림은 동작 예시이며 성능 개선 증거는 아닙니다.

**다음 행동:** 학습용 증강 설정을 바꿀 때 박스·라벨의 동반 변환부터 확인합니다.

In [ ]:
viz.plot_flip_example(ctx)

## 16. 정리
원본 분석 결과를 다시 모읍니다.

**발견한 점:** 클래스 불균형과 원본 라벨 누락이 있고, 사진·JSON 파일 자체의 짝은 맞습니다.

**다음 행동:** 정제본 처리 여부와 검증 분할을 확인한 뒤 증강·모델 실험 효과를 비교합니다.

In [ ]:
display(eda.eda_inventory(ctx))
display(eda.file_alignment(ctx))
print("누락 후보 사진:",len(missing),"/ 테스트 사진:",int(test_inventory["사진 수"].sum()))

## 부록. EDA 점검이 모델 평가에 미친 영향
**EDA에서 확인한 것:** 사진과 정답 라벨·교정표의 불일치입니다.

**모델 실험에서 확인한 것:** 저장된 예측을 같은 검증 47장의 교정 정답으로 다시 채점했습니다. 모델을 재학습하거나 새 예측을 만든 것은 아닙니다. 교정 전 점수가 원래 기록과 재현되는지 먼저 확인했습니다.

D1의 mAP@[0.75:0.95]는 0.96903→0.98341, D1+증강은 0.93817→0.95234입니다. D3 실행에는 D1의 RFS가 유지돼 순수 D3와 구분합니다. manual_fix만 적용했으며 manual_drop·자동 정제를 포함한 전체 v1 평가는 아닙니다.

**다음 행동:** 교정 정답을 공통 기준으로 정하고, 모델이 남긴 실제 혼동과 라벨 오류를 구분합니다. 이 점수는 검증 점수이며 캐글 점수·정답률이 아닙니다.

[팀 W&B의 원래 점수·교정 점수·비교 자료](https://forge.coreweave.com/wandb/YacssoK/soonwoo_yolov8m)

집계 점수는 2026-11-02 고정 기록입니다. 교정표가 바뀌면 사진 대조 표와 집계 시점이 달라질 수 있으므로 새 정답으로 재평가해야 합니다.


In [ ]:
display(eda.quality_audit_table(quality_audit, "models"))
viz.plot_quality_audit(quality_audit, ctx, "models")
